# BRSET — Fairness audit with bootstrap CIs

**Why this notebook exists:** the original BRSET training notebook computed subgroup AUROCs as point estimates only. Without confidence intervals, we can't claim findings like "drusens AUROC drops 8 points on Nikon" are real. This notebook re-runs the subgroup analysis with 1000-resample bootstrap CIs so each subgroup number comes with [lo, hi] error bars.

**No retraining.** Loads the existing checkpoint from your BRSET training notebook output.

**Inputs to attach (right sidebar → + Add Input):**
1. `samarthmishra208/brset-brazilian-multilabel-ophthalmological` (Datasets tab)
2. **Notebook Output:** your BRSET training notebook (`samarthmishra208/brset-baseline` or whatever you named it)

**Settings:** GPU T4 x2 (5 min inference) or CPU (15 min). Internet off.

**Runtime:** ~5-10 minutes total.

## Cell 1 — Imports + locate inputs

In [ ]:
import os, sys, json, time, glob, pathlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Find checkpoint + index from training notebook output
ckpt_candidates = glob.glob("/kaggle/input/**/best.pt", recursive=True)
idx_candidates  = glob.glob("/kaggle/input/**/brset_index.csv", recursive=True)
print("\nFound best.pt at:", ckpt_candidates)
print("Found brset_index.csv at:", idx_candidates)
assert ckpt_candidates, "best.pt not found — attach BRSET training notebook output"
assert idx_candidates,  "brset_index.csv not found — attach BRSET training notebook output"
CKPT = ckpt_candidates[0]
INDEX_CSV = idx_candidates[0]

# BRSET data paths
BRSET_ROOT = "/kaggle/input/datasets/samarthmishra208/brset-brazilian-multilabel-ophthalmological/a-brazilian-multilabel-ophthalmological-dataset-brset-1.0.1"
IMAGES_DIR = os.path.join(BRSET_ROOT, "fundus_photos")
assert os.path.exists(IMAGES_DIR), f"Missing images dir: {IMAGES_DIR}"

print(f"\nCKPT: {CKPT}")
print(f"INDEX_CSV: {INDEX_CSV}")
print(f"IMAGES_DIR: {IMAGES_DIR}")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)

## Cell 2 — Dataset + transforms + model definitions

In [ ]:
DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]
N_LABELS = len(DISEASE_COLS)

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class BRSETTestDataset(Dataset):
    def __init__(self, index_csv, images_dir, transform=None):
        full = pd.read_csv(index_csv)
        self.df = full[full["split"] == "test"].reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform
        self.labels = self.df[DISEASE_COLS].astype(np.float32).to_numpy()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # abs_path may be stale (from a previous notebook session) so rebuild
        img_path = os.path.join(self.images_dir, f"{row['image_id']}.jpg")
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return (img, y, str(row["patient_id"]),
                int(row.get("patient_sex", -1)),
                str(row.get("camera", "")),
                str(row.get("quality", "")),
                float(row["patient_age"]) if pd.notna(row.get("patient_age")) else -1.0)

def build_resnet50(num_labels=N_LABELS, dropout=0.3):
    m = models.resnet50(weights=None)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, num_labels))
    return m

test_ds = BRSETTestDataset(INDEX_CSV, IMAGES_DIR, transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"Test dataset: {len(test_ds):,} images")

## Cell 3 — Load checkpoint + run inference

In [ ]:
state = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = build_resnet50().to(DEVICE)
model.load_state_dict(state["model_state"]); model.eval()
print(f"Loaded checkpoint epoch {state['epoch']+1}  val_macro_auroc={state.get('val_macro_auroc', '?')}")

@torch.no_grad()
def run_inference(model, loader, device):
    all_probs, all_y, all_pids, all_sex, all_cam, all_qual, all_age = [], [], [], [], [], [], []
    for x, y, pid, sex, cam, qual, age in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
        probs = torch.sigmoid(logits.float()).cpu().numpy()
        all_probs.append(probs); all_y.append(y.numpy())
        all_pids.extend(list(pid))
        all_sex.extend([int(s) for s in sex])
        all_cam.extend(list(cam)); all_qual.extend(list(qual))
        all_age.extend([float(a) for a in age])
    return (np.concatenate(all_probs), np.concatenate(all_y),
            np.array(all_pids), np.array(all_sex),
            np.array(all_cam), np.array(all_qual), np.array(all_age))

print("\nRunning inference...")
t0 = time.time()
probs, y_true, pids, sex, cam, qual, age = run_inference(model, test_loader, DEVICE)
print(f"  {len(y_true):,} predictions in {time.time()-t0:.1f}s")

## Cell 4 — Bootstrap CI subgroup audit

For every (subgroup × disease) pair, compute AUROC + 95% CI via 1000-resample nonparametric bootstrap. Only report subgroups with ≥30 samples AND ≥10 positive labels for the disease (below that, CIs are too wide to mean anything).

In [ ]:
def bootstrap_auroc(p, y, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yi, pi = y[idx], p[idx]
        if 0 < yi.sum() < len(yi):
            aurocs.append(roc_auc_score(yi, pi))
            auprcs.append(average_precision_score(yi, pi))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return dict(
        n=int(n), n_pos=int(y.sum()),
        auroc=float(np.mean(aurocs)) if aurocs else None,
        auroc_ci95=ci(aurocs) if aurocs else None,
        auprc=float(np.mean(auprcs)) if auprcs else None,
        auprc_ci95=ci(auprcs) if auprcs else None,
    )

MIN_SUBGROUP_N = 30
MIN_POS_PER_LABEL = 10

def audit_subgroup(probs, y_true, mask, subgroup_label):
    if mask.sum() < MIN_SUBGROUP_N:
        return None
    p, y = probs[mask], y_true[mask]
    out = {"subgroup": subgroup_label, "n": int(mask.sum()), "per_label": {}}
    for i, d in enumerate(DISEASE_COLS):
        yb = y[:, i]; pb = p[:, i]
        if yb.sum() >= MIN_POS_PER_LABEL and yb.sum() < len(yb):
            out["per_label"][d] = bootstrap_auroc(pb, yb)
        else:
            out["per_label"][d] = None  # too few positives in this subgroup
    return out

print(f"\nRunning bootstrap audits (min n={MIN_SUBGROUP_N}, min pos/label={MIN_POS_PER_LABEL})...")

audit = {"overall": {}}
# OVERALL (whole test set with CIs)
for i, d in enumerate(DISEASE_COLS):
    audit["overall"][d] = bootstrap_auroc(probs[:, i], y_true[:, i])

# By sex
audit["by_sex"] = []
for s, name in [(1, "male"), (2, "female")]:
    r = audit_subgroup(probs, y_true, sex == s, f"sex={name}")
    if r: audit["by_sex"].append(r)

# By camera
audit["by_camera"] = []
for c in sorted(set(cam)):
    if not c: continue
    r = audit_subgroup(probs, y_true, cam == c, f"camera={c}")
    if r: audit["by_camera"].append(r)

# By quality
audit["by_quality"] = []
for q in sorted(set(qual)):
    if not q: continue
    r = audit_subgroup(probs, y_true, qual == q, f"quality={q}")
    if r: audit["by_quality"].append(r)

# By age band
audit["by_age_band"] = []
known_age = age >= 0
for lo, hi, name in [(0,40,"<40"),(40,60,"40-60"),(60,80,"60-80"),(80,200,"80+")]:
    mask = known_age & (age >= lo) & (age < hi)
    r = audit_subgroup(probs, y_true, mask, f"age={name}")
    if r: audit["by_age_band"].append(r)

# Save full JSON
with open(RESULTS / "fairness_audit_with_cis.json", "w") as f:
    json.dump(audit, f, indent=2)
print(f"\nSaved to {RESULTS / 'fairness_audit_with_cis.json'}")

## Cell 5 — Print human-readable summary

For each subgroup axis, show AUROC ± 95% CI for the high-prevalence labels. Subgroup-vs-subgroup gaps where CIs don't overlap are the publishable findings.

In [ ]:
def fmt(v):
    if v is None: return "—"
    return f"{v['auroc']:.3f} [{v['auroc_ci95'][0]:.3f},{v['auroc_ci95'][1]:.3f}] (n_pos={v['n_pos']})"

# High-prevalence labels worth comparing
HIGH_PREV = ["diabetic_retinopathy", "drusens", "increased_cup_disc",
             "macular_edema", "amd", "scar"]

def print_axis(rows, axis_name):
    print(f"\n=== {axis_name} ===")
    if not rows:
        print("  (no usable subgroups)"); return
    # Column headers
    header = f"  {'disease':28s} "
    for r in rows:
        label = r["subgroup"] + f" n={r['n']}"
        header += f"  {label:>40s}"
    print(header)
    for d in HIGH_PREV:
        line = f"  {d:28s} "
        for r in rows:
            v = r["per_label"].get(d)
            line += f"  {fmt(v):>40s}"
        print(line)

print("\n=== Overall (with CIs) ===")
for d in DISEASE_COLS:
    v = audit["overall"][d]
    if v: print(f"  {d:30s}  {fmt(v)}")

print_axis(audit["by_sex"],       "By sex")
print_axis(audit["by_camera"],    "By camera")
print_axis(audit["by_quality"],   "By quality")
print_axis(audit["by_age_band"],  "By age band")

# Identify subgroup gaps where CIs don't overlap (publishable findings)
def find_gaps(rows, axis_name):
    if len(rows) < 2: return
    print(f"\n=== Subgroup gaps where 95% CIs don't overlap ({axis_name}) ===")
    found = False
    for d in DISEASE_COLS:
        results = []
        for r in rows:
            v = r["per_label"].get(d)
            if v: results.append((r["subgroup"], v["auroc"], v["auroc_ci95"]))
        if len(results) < 2: continue
        for i in range(len(results)):
            for j in range(i+1, len(results)):
                a, b = results[i], results[j]
                # CIs don't overlap if a_hi < b_lo OR b_hi < a_lo
                if a[2][1] < b[2][0] or b[2][1] < a[2][0]:
                    found = True
                    print(f"  {d:30s}  {a[0]}: {a[1]:.3f} {a[2]}  vs  {b[0]}: {b[1]:.3f} {b[2]}")
    if not found:
        print("  (no statistically significant gaps found)")

find_gaps(audit["by_sex"],      "sex")
find_gaps(audit["by_camera"],   "camera")
find_gaps(audit["by_quality"],  "quality")
find_gaps(audit["by_age_band"], "age band")


## Cell 6 — Persist artifacts

In [ ]:
import shutil
zip_path = WORK / "brset_audit_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size/1e3:.1f} KB)")
print("Download from Notebook page → Output tab → brset_audit_results.zip")